# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step tutorial for loading and exploring the 
FAIR² dataset using the `mlcroissant` library. We demonstrate how to load the metadata, 
investigate the available record sets, and perform exploratory analyses—always referencing 
entities by their unique `@id` as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Loaded dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.
Here we print out the available record sets, their `@id`, and available fields. We will use these `@id`s in all following operations.

In [ ]:
# List all record sets and their @id
record_sets = []
for record_set in dataset.record_sets():
    print(f"Record Set: {record_set.name} [@id: {record_set.id}]")
    record_sets.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} [@id: {field.id}] (dataType: {field.data_type})")
    print()

# For demonstration, print first few records from the first record set
if record_sets:
    rs_id_demo = record_sets[0]
    print(f"First 2 records for RecordSet @id={rs_id_demo}:")
    for i, rec in enumerate(dataset.records(record_set=rs_id_demo)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Fields and record sets are referenced by their `@id` as seen above.

In [ ]:
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"RecordSet @id: {rs_id}, Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"RecordSet @id: {rs_id} is empty.")

## 4. Exploratory Data Analysis (EDA)
You can now process and analyze your data! First, select a numeric field and a grouping field by their unique `@id`.
Below, we filter, normalize, and perform basic grouping—all referencing fields via their `@id`.

In [ ]:
# Choose a RecordSet by @id
# (You may wish to change the below IDs depending on what was printed in the overview above)
record_set_id = record_sets[0]  # Change if you want to analyze another table
df = dataframes[record_set_id]

# Identify numeric fields (by @id) from Data Overview above.
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in {record_set_id}: {numeric_fields}")
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Pick first numeric field by @id
else:
    numeric_field_id = df.columns[0]  # Fallback: Use first field

# Filtering (e.g., keep rows with values > threshold)
threshold = 10  # You may adjust this threshold per your data
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

# Normalization
if len(filtered_df) >= 2 and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a categorical field (by @id)
group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id]
if group_fields:
    group_field_id = group_fields[0]
    print(f"\nGrouping by '{group_field_id}':")
    grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
    print(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.show()

# If grouping possible, show boxplot
if group_fields and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load dataset metadata and records using the `mlcroissant` library from a FAIR Croissant schema.
- Identify and reference entities by their unique `@id` (including record sets and fields).
- Extract and process data for exploratory analysis (filtering, normalization, grouping).
- Visualize selected columns.

You can now further analyze, model, or export the data as needed for your science or data science tasks.